# Loading the libraries and Data files

In [1]:
import pandas as pd
import numpy as np

mql = pd.read_csv("olist_marketing_qualified_leads_dataset.csv")
deals = pd.read_csv("olist_closed_deals_dataset.csv")

mql["first_contact_date"] = pd.to_datetime(mql["first_contact_date"])
deals["won_date"] = pd.to_datetime(deals["won_date"])

funnel = mql.merge(deals, on="mql_id", how="left", indicator=True)
funnel["converted"] = (funnel["_merge"] == "both").astype(int)

# 1. Monthly funnel trend

In [2]:
monthly_funnel = (
    funnel.assign(contact_month=funnel["first_contact_date"].dt.to_period("M").astype(str))
    .groupby("contact_month", as_index=False)
    .agg(
        mqls=("mql_id", "count"),
        closed_deals=("converted", "sum")
    )
)

monthly_funnel["conversion_rate_pct"] = (
    100 * monthly_funnel["closed_deals"] / monthly_funnel["mqls"]
).round(2)

print(monthly_funnel)

   contact_month  mqls  closed_deals  conversion_rate_pct
0        2017-06     4             0                 0.00
1        2017-07   239             2                 0.84
2        2017-08   386             9                 2.33
3        2017-09   312             7                 2.24
4        2017-10   416            14                 3.37
5        2017-11   445            18                 4.04
6        2017-12   200            11                 5.50
7        2018-01  1141           152                13.32
8        2018-02  1028           149                14.49
9        2018-03  1174           167                14.22
10       2018-04  1352           183                13.54
11       2018-05  1303           130                 9.98


# 2. Time-to-close distribution

In [3]:
won = funnel[funnel["converted"] == 1].copy()

won["days_to_close"] = (
    won["won_date"] - won["first_contact_date"]
).dt.total_seconds() / 86_400

won = won[won["days_to_close"] >= 0].copy()

bins = [-1, 7, 30, 60, 90, np.inf]
labels = ["0–7 days", "8–30 days", "31–60 days", "61–90 days", "91+ days"]

won["close_time_band"] = pd.cut(
    won["days_to_close"],
    bins=bins,
    labels=labels
)

close_time_distribution = (
    won.groupby("close_time_band", observed=False)
    .size()
    .reset_index(name="deals")
)

close_time_distribution["share_of_deals_pct"] = (
    100 * close_time_distribution["deals"]
    / close_time_distribution["deals"].sum()
).round(2)

print(close_time_distribution)

  close_time_band  deals  share_of_deals_pct
0        0–7 days    248               29.49
1       8–30 days    298               35.43
2      31–60 days     97               11.53
3      61–90 days     47                5.59
4        91+ days    151               17.95


# 3. Best origin × landing-page combinations

In [4]:
funnel["origin_clean"] = (
    funnel["origin"]
    .fillna("")
    .str.strip()
    .replace("", "(blank)")
)

origin_landing_pages = (
    funnel.groupby(["origin_clean", "landing_page_id"], as_index=False)
    .agg(
        mqls=("mql_id", "count"),
        closed_deals=("converted", "sum")
    )
)

origin_landing_pages["conversion_rate_pct"] = (
    100 * origin_landing_pages["closed_deals"]
    / origin_landing_pages["mqls"]
).round(2)

top_origin_landing_pages = (
    origin_landing_pages[origin_landing_pages["mqls"] >= 20]
    .sort_values(
        ["conversion_rate_pct", "mqls"],
        ascending=[False, False]
    )
    .head(25)
)

print(top_origin_landing_pages)

        origin_clean                   landing_page_id  mqls  closed_deals  \
140   direct_traffic  ce1a65abd0973638f1c887a6efcfa82d    46            11   
376   organic_search  30077c17f2ec5010a82e37ad8925b95f    21             5   
832      paid_search  fbc24da54d531c6204ae2d17b1090bb1    21             5   
369   organic_search  22c29808c4f815213303f8933030604c   495           112   
397   organic_search  40dec9f3d5259a3d2dbcdab2114fae47    57            12   
699      paid_search  40dec9f3d5259a3d2dbcdab2114fae47   241            50   
500   organic_search  b76ef37428e6799c421989521c0e5077   116            24   
455   organic_search  7fa6214d82e911d070f51ef79381b956    29             6   
1151         unknown  b76ef37428e6799c421989521c0e5077   656           134   
792      paid_search  ce1a65abd0973638f1c887a6efcfa82d    51            10   
1045         unknown  22c29808c4f815213303f8933030604c    36             7   
852         referral  22c29808c4f815213303f8933030604c    56    

# 4. SDR and sales-rep closed-deal workload

In [5]:
rep_performance = (
    won.groupby(["sdr_id", "sr_id"], as_index=False)
    .agg(
        closed_deals=("mql_id", "count"),
        unique_sellers=("seller_id", "nunique"),
        avg_days_to_close=("days_to_close", "mean")
    )
)

rep_performance["avg_days_to_close"] = (
    rep_performance["avg_days_to_close"].round(1)
)

rep_performance = rep_performance.sort_values(
    ["closed_deals", "avg_days_to_close"],
    ascending=[False, True]
)

print(rep_performance)

                               sdr_id                             sr_id  \
94   4b339f9567d060bcea4f5136b9f5949e  4ef15afb4b2723d8f3d81e51ec7afefe   
187  de63de0d10a6012430098db33c679b0b  d3d1e91a157ea7f90548eef82f1955e3   
96   4b339f9567d060bcea4f5136b9f5949e  6565aa9ce3178a5caf6171827af3a9ba   
110  56bf83c4bb35763a51c2baab501b4c67  4ef15afb4b2723d8f3d81e51ec7afefe   
183  de63de0d10a6012430098db33c679b0b  4ef15afb4b2723d8f3d81e51ec7afefe   
..                                ...                               ...   
87   45749fb708130f78d0db07d8d80f030b  9d12ef1a7eca3ec58c545c678af7869c   
84   45749fb708130f78d0db07d8d80f030b  4b339f9567d060bcea4f5136b9f5949e   
120  6aa3b86a83d784b05f0e37e26b20860d  56bf83c4bb35763a51c2baab501b4c67   
168  b34f6eba10f46bf9a657a01c108a8284  a8387c01a09e99ce014107505b92388c   
47   2b63542749aa9caf15f21816da1db341  d3d1e91a157ea7f90548eef82f1955e3   

     closed_deals  unique_sellers  avg_days_to_close  
94             21              21           

# 5. Closed-deal profile by segment and lead type

In [6]:
deal_profile = (
    deals.groupby(["business_segment", "lead_type"], dropna=False)
    .agg(
        closed_deals=("mql_id", "count"),
        unique_sellers=("seller_id", "nunique")
    )
    .reset_index()
    .sort_values("closed_deals", ascending=False)
)

deal_profile["share_of_deals_pct"] = (
    100 * deal_profile["closed_deals"] / deal_profile["closed_deals"].sum()
).round(2)

print(deal_profile.head(20))

                    business_segment      lead_type  closed_deals  \
95                        home_decor  online_medium            44   
84                     health_beauty  online_medium            37   
34                   car_accessories  online_medium            36   
6            audio_video_electronics  online_medium            31   
108              household_utilities  online_medium            28   
49   construction_tools_house_garden  online_medium            24   
94                        home_decor     online_big            17   
48   construction_tools_house_garden     online_big            17   
33                   car_accessories     online_big            15   
83                     health_beauty     online_big            15   
147                   sports_leisure  online_medium            14   
104              household_utilities       industry            14   
81                     health_beauty        offline            13   
41                         compute

# 6. Monthly conversion trend by acquisition origin

In [7]:
funnel["contact_month"] = funnel["first_contact_date"].dt.to_period("M").astype(str)

monthly_origin_conversion = (
    funnel.groupby(["contact_month", "origin_clean"], as_index=False)
    .agg(
        mqls=("mql_id", "count"),
        closed_deals=("converted", "sum")
    )
)

monthly_origin_conversion["conversion_rate_pct"] = (
    100
    * monthly_origin_conversion["closed_deals"]
    / monthly_origin_conversion["mqls"]
).round(2)

monthly_origin_conversion = monthly_origin_conversion.sort_values(
    ["contact_month", "conversion_rate_pct"],
    ascending=[True, False]
)

print(monthly_origin_conversion.head(30))

   contact_month       origin_clean  mqls  closed_deals  conversion_rate_pct
0        2017-06            (blank)     1             0                 0.00
1        2017-06            display     1             0                 0.00
2        2017-06              email     1             0                 0.00
3        2017-06            unknown     1             0                 0.00
4        2017-07            (blank)     5             1                20.00
14       2017-07            unknown    35             1                 2.86
5        2017-07     direct_traffic    16             0                 0.00
6        2017-07            display    19             0                 0.00
7        2017-07              email    19             0                 0.00
8        2017-07     organic_search    54             0                 0.00
9        2017-07              other    12             0                 0.00
10       2017-07  other_publicities     3             0                 0.00

# 7. Landing-page performance analysis

In [8]:
landing_page_performance = (
    funnel.groupby("landing_page_id", as_index=False)
    .agg(
        mqls=("mql_id", "count"),
        closed_deals=("converted", "sum")
    )
)

landing_page_performance["conversion_rate_pct"] = (
    100
    * landing_page_performance["closed_deals"]
    / landing_page_performance["mqls"]
).round(2)

# Prevents small landing pages from appearing as false “top performers”
best_landing_pages = (
    landing_page_performance
    .query("mqls >= 30")
    .sort_values(
        ["conversion_rate_pct", "mqls"],
        ascending=[False, False]
    )
)

print(best_landing_pages.head(15))

                      landing_page_id  mqls  closed_deals  conversion_rate_pct
77   30077c17f2ec5010a82e37ad8925b95f    48            10                20.83
122  40dec9f3d5259a3d2dbcdab2114fae47   330            67                20.30
61   22c29808c4f815213303f8933030604c   883           174                19.71
356  b76ef37428e6799c421989521c0e5077   912           171                18.75
417  d83b0d0e48c8447d1d5507a44027a955    34             6                17.65
240  7fa6214d82e911d070f51ef79381b956    68            11                16.18
394  ce1a65abd0973638f1c887a6efcfa82d   394            59                14.97
345  b48ec5f3b04e9068441002a19df93c6c    51             7                13.73
94   35c9b150ab36fe584c1f24fd458c453a    59             8                13.56
62   241f79c7a8fe0270f4fb79fcbbcd17ad   109            14                12.84
46   1ceb590cd1e00c7ee95220971f82693d    71             9                12.68
2    0218f6be0b76aca72ab4d00ee9e8cf10    48         

# 8. Data-quality and missing-value analysis

In [9]:
deal_columns_to_check = [
    "business_segment",
    "lead_type",
    "lead_behaviour_profile",
    "has_company",
    "has_gtin",
    "average_stock",
    "business_type",
    "declared_product_catalog_size",
    "declared_monthly_revenue"
]

data_quality = pd.DataFrame({
    "column": deal_columns_to_check,
    "blank_or_missing_count": [
        deals[column].isna().sum()
        + deals[column].fillna("").astype(str).str.strip().eq("").sum()
        for column in deal_columns_to_check
    ]
})

data_quality["blank_or_missing_pct"] = (
    100 * data_quality["blank_or_missing_count"] / len(deals)
).round(2)

data_quality = data_quality.sort_values(
    "blank_or_missing_pct",
    ascending=False
)

print(data_quality)

                          column  blank_or_missing_count  blank_or_missing_pct
3                    has_company                    1558                185.04
4                       has_gtin                    1556                184.80
5                  average_stock                    1552                184.32
7  declared_product_catalog_size                    1546                183.61
2         lead_behaviour_profile                     354                 42.04
6                  business_type                      20                  2.38
1                      lead_type                      12                  1.43
0               business_segment                       2                  0.24
8       declared_monthly_revenue                       0                  0.00


# 9. ML: predict likelihood that an MQL becomes a closed deal
This model uses only fields available at the MQL stage: origin, landing_page_id, and contact month. It does not use closed-deal fields such as segment, rep, revenue, or won date.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

ml_data = funnel[
    ["mql_id", "first_contact_date", "origin_clean", "landing_page_id", "converted"]
].copy()

ml_data["contact_month"] = (
    ml_data["first_contact_date"].dt.to_period("M").astype(str)
)

# Chronological split: older leads train the model, most recent 20% evaluate it.
ml_data = ml_data.sort_values("first_contact_date").reset_index(drop=True)
split_index = int(len(ml_data) * 0.80)

train = ml_data.iloc[:split_index].copy()
test = ml_data.iloc[split_index:].copy()

# Group rare landing pages to reduce overfitting.
common_pages = train["landing_page_id"].value_counts()
common_pages = common_pages[common_pages >= 10].index

train["landing_page_group"] = np.where(
    train["landing_page_id"].isin(common_pages),
    train["landing_page_id"],
    "other_or_rare_page"
)

test["landing_page_group"] = np.where(
    test["landing_page_id"].isin(common_pages),
    test["landing_page_id"],
    "other_or_rare_page"
)

features = ["origin_clean", "landing_page_group", "contact_month"]
target = "converted"

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical_features",
            OneHotEncoder(handle_unknown="ignore"),
            features
        )
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

model.fit(train[features], train[target])

test["predicted_conversion_probability"] = model.predict_proba(
    test[features]
)[:, 1]

test["predicted_conversion"] = (
    test["predicted_conversion_probability"] >= 0.50
).astype(int)

auc = roc_auc_score(
    test[target],
    test["predicted_conversion_probability"]
)

print(f"Test ROC-AUC: {auc:.3f}")
print(
    classification_report(
        test[target],
        test["predicted_conversion"],
        zero_division=0
    )
)

Test ROC-AUC: 0.651
              precision    recall  f1-score   support

           0       0.91      0.87      0.89      1444
           1       0.13      0.18      0.15       156

    accuracy                           0.80      1600
   macro avg       0.52      0.52      0.52      1600
weighted avg       0.83      0.80      0.82      1600



# 10. Lead-priority list and data-based recommendations

In [11]:
lead_priority_list = (
    test[
        [
            "mql_id",
            "first_contact_date",
            "origin_clean",
            "landing_page_id",
            "predicted_conversion_probability",
            "converted"
        ]
    ]
    .sort_values("predicted_conversion_probability", ascending=False)
    .reset_index(drop=True)
)

print("\nTop predicted MQLs:")
print(lead_priority_list.head(20))

# Channel performance for recommendations
channel_summary = (
    funnel.groupby("origin_clean", as_index=False)
    .agg(
        mqls=("mql_id", "count"),
        closed_deals=("converted", "sum")
    )
)

channel_summary["conversion_rate_pct"] = (
    100 * channel_summary["closed_deals"] / channel_summary["mqls"]
).round(2)

reliable_channels = channel_summary.query("mqls >= 100").copy()

best_channel = reliable_channels.sort_values(
    "conversion_rate_pct",
    ascending=False
).iloc[0]

weak_channel = reliable_channels.sort_values(
    "conversion_rate_pct",
    ascending=True
).iloc[0]

overall_conversion = 100 * funnel["converted"].mean()

print("\n--- Recommendations ---")
print(
    f"1. Overall MQL-to-closed-deal conversion is "
    f"{overall_conversion:.2f}%."
)
print(
    f"2. Prioritize or investigate '{best_channel['origin_clean']}', "
    f"which has the strongest observed conversion among channels with "
    f"at least 100 MQLs: {best_channel['conversion_rate_pct']:.2f}%."
)
print(
    f"3. Review '{weak_channel['origin_clean']}' before increasing spend. "
    f"It has {weak_channel['mqls']} MQLs but only a "
    f"{weak_channel['conversion_rate_pct']:.2f}% observed conversion rate."
)
print(
    "4. Use predicted conversion probability to prioritize SDR follow-up, "
    "but validate the model regularly because channel and landing-page "
    "performance can change over time."
)
print(
    "5. Improve data collection for company, GTIN, stock, catalog size, "
    "and revenue before using those fields in lead-scoring models."
)
print(
    "6. Do not use this model as an automatic rejection rule; use it to "
    "prioritize outreach and test whether high-scoring leads close faster."
)


Top predicted MQLs:
                              mql_id first_contact_date origin_clean  \
0   717d08a84121310b4393bb0f398377e0         2018-04-25      unknown   
1   15637ed71e3e1c7523d8f4f2160a71d6         2018-04-25      unknown   
2   66ff0b5dbb060ef47f756d33cf170f97         2018-04-30      (blank)   
3   f5baaf0afe419681731ec3d30dafd954         2018-04-30      (blank)   
4   e9fb2eda3d9c55a0d89c98d6c54b5b3e         2018-04-27      (blank)   
5   5bb0b4dd9a4f5c0de3564b63d74bcf5f         2018-04-24  paid_search   
6   5bfcd83ad8acb6ab91ffa9a89da30c90         2018-04-28  paid_search   
7   5874f1b550c0d4cf4a1774b46b8d5398         2018-04-27  paid_search   
8   22c0b2a19a05fff19e92fab5a0b7728b         2018-04-25  paid_search   
9   f36c682dbd97d51bdc2e0cfd81ea3028         2018-04-26  paid_search   
10  d6beac7184414a6487a7edade4c0f2ef         2018-04-27  paid_search   
11  66a18289224fcaca68694426ede468a0         2018-04-26  paid_search   
12  6107eb48aa5ee13a438f2c0e903de38a       